# Data Augmentation at Scale — BioCatch Vishing Dataset (1M rows)



## Objective

Scale the original dataset of 50,000 sessions to **1,000,000 sessions** with a realistic vishing proportion (1-2%), simulating an environment closer to production where:

- The data volume is high (~1M sessions)
- The imbalance is severe (~1-2% fraud, as in reality)
- The distributions and correlations of the data are faithfully preserved

### Augmentation Strategy

Unlike SMOTE (which interpolates between neighbors), we use a **parametric augmentation strategy with controlled perturbation** that:

1. **Estimates the distribution of each feature** per class (mean, std, empirical distribution)
2. **Generates new samples using bootstrap with Gaussian perturbation** — takes a real sample and adds noise proportional to the local variability
3. **Preserves correlations** using Cholesky decomposition on the original correlation matrix
4. **Respects domain constraints** (ranges, integer types, binary variables, non-negativity)
5. **Injects temporal and customer variability** to simulate seasonal patterns and user diversity

### Output

A dataset of 1,000,000 rows with is_vishing at ~1.5%, ready to train models under realistic conditions.


## 1. Setup and Data Loading

In [ ]:
!pip install -r "requirements.txt"

In [ ]:
import boto3
import sagemaker

# Identify session and role
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as scipy_stats
import pyarrow
import fastparquet

plt.rcParams.update({
    'figure.figsize': (14, 6), 'font.size': 11, 'axes.titlesize': 14,
    'axes.labelsize': 12, 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
})

COLORS = {'legit': '#2ecc71', 'vishing': '#e74c3c', 'neutral': '#3498db'}
np.random.seed(42)

print('✅ Libraries loaded')

In [ ]:
s3_input_path = 's3://poc-fraude-vishing/biocatch_sinthetic_data.csv'
df_original = pd.read_csv(s3_input_path, parse_dates=['session_timestamp'])


print(f'Original dataset: {len(df_original):,} rows × {df_original.shape[1]} columns')
print(f'  Legitimate: {(df_original["is_vishing"]==0).sum():,}')
print(f'  Vishing:    {(df_original["is_vishing"]==1).sum():,} ({df_original["is_vishing"].mean()*100:.1f}%)')

## 2. Feature Classification by Type

In [ ]:
# Metadata columns (not augmented, they are regenerated)
meta_cols = ['session_id', 'customer_id', 'session_timestamp',
             'device_type', 'os_type', 'app_version']

# Label and claim metadata
label_cols = ['is_vishing', 'days_to_claim', 'claim_category']

# Continuous features
continuous_cols = [
    'avg_keyhold_ms', 'avg_interkey_latency_ms', 'typing_speed_cps',
    'keystroke_variability', 'segmented_typing_ratio',
    'avg_touch_pressure', 'avg_touch_size_px', 'swipe_speed_px_s',
    'swipe_directional_variance', 'scroll_speed_avg',
    'device_tilt_angle_mean', 'device_tilt_variability',
    'gyro_rotation_rate_mean', 'accelerometer_jerk_mean',
    'avg_hesitation_duration_s', 'max_hesitation_duration_s',
    'total_dead_time_s', 'dead_time_ratio',
    'screen_transition_time_avg_s', 'data_familiarity_score',
    'session_duration_s', 'call_overlap_duration_s',
    'time_to_transaction_s',
    'errors_per_minute', 'interactions_per_s', 'hesitation_composite',
]

# Integer features (count-based)
integer_cols = [
    'phone_motion_events', 'hesitation_count', 'dead_time_periods',
    'screens_visited', 'unique_screens_visited', 'unusual_screen_visits',
    'navigation_back_count', 'input_error_count', 'input_correction_count',
    'amount_field_corrections', 'beneficiary_field_corrections',
    'copy_paste_events', 'doodling_events', 'hour_of_day',
    'transaction_amount_cop',
    'biocatch_risk_score', 'biocatch_genuine_score',
]

# Binary features
binary_cols = [
    'is_atypical_hour', 'phone_call_active',
    'remote_access_tool_detected', 'suspicious_app_detected',
    'transaction_attempted', 'is_new_beneficiary',
    'biocatch_ato_indicator', 'biocatch_social_eng_indicator',
    'biocatch_bot_indicator',
]

# Features with range [0, 1]
ratio_cols = ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
              'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']

# All numeric features
all_feature_cols = continuous_cols + integer_cols + binary_cols

print(f'Continuous features: {len(continuous_cols)}')
print(f'Integer features:    {len(integer_cols)}')
print(f'Binary features:     {len(binary_cols)}')
print(f'Total features:      {len(all_feature_cols)}')

## 3. Augmentation Strategy: Correlated Bootstrap with Perturbation

The key for a million rows to make sense is preserving the multivariate structure of the data, not just the marginal distributions. The strategy is:

1. **Separate by class** (legitimate vs vishing) — each class is augmented independently
2. **Estimate the correlation matrix** of each class over the continuous features
3. **Generate correlated data** using Cholesky: independent Gaussian noise is generated and transformed with the original correlation structure
4. **Map to real marginal distributions** using inverse transform sampling — each feature is fit to the empirical distribution of the original class
5. **Post-process** to respect types, ranges, and logical consistencies

In [ ]:
class CorrelatedAugmenter:
    """
    Generates synthetic data that preserves:
    - Marginal distributions of each feature (via quantile matching)
    - Correlation structure between features (via Cholesky)
    - Domain constraints (types, ranges, non-negativity)
    """
    
    def __init__(self, df_class, continuous_cols, integer_cols, binary_cols):
        self.continuous_cols = continuous_cols
        self.integer_cols = integer_cols
        self.binary_cols = binary_cols
        self.all_cols = continuous_cols + integer_cols + binary_cols
        
        # Store original data for bootstrap and statistics
        self.df_class = df_class[self.all_cols].copy()
        self.n_original = len(df_class)
        
        # Compute statistics per feature
        self.means = self.df_class.mean()
        self.stds = self.df_class.std()
        self.mins = self.df_class.min()
        self.maxs = self.df_class.max()
        
        # Probabilities of the binary features
        self.binary_probs = {col: df_class[col].mean() for col in binary_cols}
        
        # Empirical distributions for integer cols
        self.integer_distributions = {}
        for col in integer_cols:
            vals = df_class[col].values
            unique, counts = np.unique(vals, return_counts=True)
            self.integer_distributions[col] = (unique, counts / counts.sum())
        
        # Correlation matrix and Cholesky decomposition for continuous features
        cont_data = df_class[continuous_cols].values
        self.corr_matrix = np.corrcoef(cont_data.T)
        # Regularize to ensure positive definite
        self.corr_matrix += np.eye(len(continuous_cols)) * 1e-6
        self.cholesky = np.linalg.cholesky(self.corr_matrix)
        
        # Store percentiles for inverse transform
        self.percentiles = {}
        for i, col in enumerate(continuous_cols):
            sorted_vals = np.sort(cont_data[:, i])
            self.percentiles[col] = sorted_vals
    
    def generate(self, n_samples, noise_scale=0.03):
        """
        Generates n_samples synthetic rows.
        
        noise_scale: controls how much variation is added.
            0.0 = exact replicas (pure bootstrap)
            0.03 = subtle perturbation (recommended)
            0.10 = high perturbation (more diversity, less fidelity)
        """
        result = pd.DataFrame(index=range(n_samples))
        
        # === CONTINUOUS: Cholesky + quantile matching ===
        # Generate correlated Gaussian noise
        z = np.random.normal(0, 1, (n_samples, len(self.continuous_cols)))
        z_correlated = z @ self.cholesky.T
        
        # Convert to uniform via normal CDF
        u = scipy_stats.norm.cdf(z_correlated)
        
        # Inverse transform: map uniform to empirical distribution
        for i, col in enumerate(self.continuous_cols):
            sorted_vals = self.percentiles[col]
            n_orig = len(sorted_vals)
            # Map u[0,1] to index in sorted_vals
            indices = np.clip((u[:, i] * n_orig).astype(int), 0, n_orig - 1)
            values = sorted_vals[indices]
            
            # Add Gaussian perturbation proportional to the local std
            local_std = self.stds[col] * noise_scale
            values = values + np.random.normal(0, local_std, n_samples)
            
            result[col] = values
        
        # === INTEGER: empirical distribution sampling + perturbation ===
        for col in self.integer_cols:
            unique, probs = self.integer_distributions[col]
            # Sample from the empirical distribution
            sampled = np.random.choice(unique, size=n_samples, p=probs)
            # Perturbation: ±1 with probability proportional to noise_scale
            perturbation = np.random.choice([-1, 0, 0, 0, 1], size=n_samples)
            mask = np.random.random(n_samples) < noise_scale * 5
            sampled = sampled + perturbation * mask
            result[col] = sampled
        
        # === BINARY: Bernoulli sampling with original probabilities ===
        for col in self.binary_cols:
            prob = self.binary_probs[col]
            result[col] = (np.random.random(n_samples) < prob).astype(int)
        
        # === POST-PROCESSING ===
        self._enforce_constraints(result)
        
        return result
    
    def _enforce_constraints(self, df):
        """Applies domain constraints."""
        # Ratios [0, 1]
        for col in ['segmented_typing_ratio', 'avg_touch_pressure', 'dead_time_ratio',
                     'data_familiarity_score', 'keystroke_variability', 'swipe_directional_variance']:
            if col in df.columns:
                df[col] = df[col].clip(0, 1)
        
        # Non-negative integers
        for col in self.integer_cols:
            if col in df.columns:
                df[col] = df[col].round().clip(lower=0).astype(int)
        
        # Binary
        for col in self.binary_cols:
            if col in df.columns:
                df[col] = df[col].round().clip(0, 1).astype(int)
        
        # BioCatch scores [0, 1000]
        for col in ['biocatch_risk_score', 'biocatch_genuine_score']:
            if col in df.columns:
                df[col] = df[col].clip(0, 1000)
        
        # Hour of day [0, 23]
        if 'hour_of_day' in df.columns:
            df['hour_of_day'] = df['hour_of_day'].clip(0, 23)
        
        # Tilt angle [0, 90]
        if 'device_tilt_angle_mean' in df.columns:
            df['device_tilt_angle_mean'] = df['device_tilt_angle_mean'].clip(0, 90)
        
        # Non-negative for all continuous features that were originally so
        for col in self.continuous_cols:
            if col in df.columns and self.mins[col] >= 0:
                df[col] = df[col].clip(lower=0)
        
        # Logical consistencies
        if 'unique_screens_visited' in df.columns and 'screens_visited' in df.columns:
            df['unique_screens_visited'] = np.minimum(
                df['unique_screens_visited'], df['screens_visited'])
        
        if 'call_overlap_duration_s' in df.columns and 'phone_call_active' in df.columns:
            df.loc[df['phone_call_active'] == 0, 'call_overlap_duration_s'] = 0
        
        if 'time_to_transaction_s' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'time_to_transaction_s'] = 0
        
        if 'transaction_amount_cop' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'transaction_amount_cop'] = 0
        
        if 'is_new_beneficiary' in df.columns and 'transaction_attempted' in df.columns:
            df.loc[df['transaction_attempted'] == 0, 'is_new_beneficiary'] = 0


print('✅ CorrelatedAugmenter defined')

## 4. Generation of the Augmented Dataset (1,000,000 sessions)

Target configuration:
- **Total**: 1,000,000 sessions
- **is_vishing=1**: ~1.5% (~15,000 sessions)
- **is_vishing=0**: ~98.5% (~985,000 sessions)
- All 50,000 original sessions are included intact

In [ ]:
# Configuration
N_TOTAL = 1_000_000
VISHING_PCT = 0.015  # 1.5%

n_vishing_target = int(N_TOTAL * VISHING_PCT)
n_legit_target = N_TOTAL - n_vishing_target

# Original data per class
df_orig_legit = df_original[df_original['is_vishing'] == 0]
df_orig_vishing = df_original[df_original['is_vishing'] == 1]

n_legit_to_generate = n_legit_target - len(df_orig_legit)
n_vishing_to_generate = n_vishing_target - len(df_orig_vishing)

print('GENERATION PLAN')
print('='*60)
print(f'Target total:           {N_TOTAL:>12,}')
print(f'Target vishing ({VISHING_PCT*100}%): {n_vishing_target:>12,}')
print(f'Target legitimate:      {n_legit_target:>12,}')
print(f'\nOriginal legitimate:    {len(df_orig_legit):>12,}')
print(f'Original vishing:       {len(df_orig_vishing):>12,}')
print(f'\nTO GENERATE legitimate: {n_legit_to_generate:>12,}')
print(f'TO GENERATE vishing:    {n_vishing_to_generate:>12,}')
print(f'\nLegitimate multiplier:  {n_legit_target/len(df_orig_legit):.1f}x')
print(f'Vishing multiplier:     {n_vishing_target/len(df_orig_vishing):.1f}x')

In [ ]:
# Create augmenters per class
print('Building augmenter for LEGITIMATE class...')
t0 = time.time()
aug_legit = CorrelatedAugmenter(df_orig_legit, continuous_cols, integer_cols, binary_cols)
print(f'  ✅ Ready ({time.time()-t0:.1f}s)')

print('\nBuilding augmenter for VISHING class...')
t0 = time.time()
aug_vishing = CorrelatedAugmenter(df_orig_vishing, continuous_cols, integer_cols, binary_cols)
print(f'  ✅ Ready ({time.time()-t0:.1f}s)')

In [ ]:
# Generate synthetic data
print('Generating synthetic LEGITIMATE class data...')
t0 = time.time()
df_synth_legit = aug_legit.generate(n_legit_to_generate, noise_scale=0.03)
print(f'  ✅ {len(df_synth_legit):,} rows in {time.time()-t0:.1f}s')

print('\nGenerating synthetic VISHING class data...')
t0 = time.time()
df_synth_vishing = aug_vishing.generate(n_vishing_to_generate, noise_scale=0.03)
print(f'  ✅ {len(df_synth_vishing):,} rows in {time.time()-t0:.1f}s')

In [ ]:
# Assemble the full dataset
print('Assembling final dataset...')
t0 = time.time()

# Originals
df_orig_legit_features = df_orig_legit[all_feature_cols].copy()
df_orig_legit_features['is_vishing'] = 0
df_orig_legit_features['is_synthetic'] = 0

df_orig_vishing_features = df_orig_vishing[all_feature_cols].copy()
df_orig_vishing_features['is_vishing'] = 1
df_orig_vishing_features['is_synthetic'] = 0

# Synthetic
df_synth_legit['is_vishing'] = 0
df_synth_legit['is_synthetic'] = 1

df_synth_vishing['is_vishing'] = 1
df_synth_vishing['is_synthetic'] = 1

# Concatenate
df_augmented = pd.concat([
    df_orig_legit_features,
    df_orig_vishing_features,
    df_synth_legit,
    df_synth_vishing,
], ignore_index=True)

# Shuffle
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

# Generate IDs
df_augmented.insert(0, 'session_id', [f'SES-{i:07d}' for i in range(1, len(df_augmented)+1)])
df_augmented.insert(1, 'customer_id', [f'CUS-{np.random.randint(10000, 99999)}' for _ in range(len(df_augmented))])

# Generate timestamps distributed over 12 months
from datetime import datetime, timedelta
base_date = datetime(2024, 6, 1)
timestamps = [base_date + timedelta(
    days=np.random.randint(0, 365),
    hours=int(df_augmented.iloc[i]['hour_of_day']),
    minutes=np.random.randint(0, 60),
    seconds=np.random.randint(0, 60)
) for i in range(len(df_augmented))]
df_augmented.insert(2, 'session_timestamp', timestamps)

# Device metadata
df_augmented.insert(3, 'device_type', np.where(np.random.rand(len(df_augmented)) < 0.85, 'mobile', 'web'))
df_augmented.insert(4, 'os_type', np.where(
    df_augmented['device_type'] == 'mobile',
    np.where(np.random.rand(len(df_augmented)) < 0.6, 'Android', 'iOS'),
    np.where(np.random.rand(len(df_augmented)) < 0.7, 'Windows', 'macOS')))
df_augmented.insert(5, 'app_version', np.random.choice(
    ['v12.1', 'v12.2', 'v12.3', 'v13.0', 'v13.1'], len(df_augmented)))

# Claim metadata
df_augmented['days_to_claim'] = df_augmented['is_vishing'].apply(
    lambda v: int(np.clip(np.random.exponential(3), 0, 30)) if v == 1 else -1)
df_augmented['claim_category'] = df_augmented['is_vishing'].apply(
    lambda v: np.random.choice(['vishing', 'ingenieria_social_telefonica', 'fraude_telefono'],
                               p=[0.6, 0.3, 0.1]) if v == 1 else 'none')

elapsed = time.time() - t0
print(f'\n✅ Dataset assembled in {elapsed:.1f}s')
print(f'   Rows: {len(df_augmented):,}')
print(f'   Columns: {df_augmented.shape[1]}')
print(f'   Vishing: {(df_augmented["is_vishing"]==1).sum():,} ({df_augmented["is_vishing"].mean()*100:.2f}%)')
print(f'   Synthetic: {(df_augmented["is_synthetic"]==1).sum():,} ({df_augmented["is_synthetic"].mean()*100:.1f}%)')

In [ ]:
# Save augmented dataset
#os.makedirs('data/augmented', exist_ok=True)
output_path = 's3://poc-fraude-vishing/data/augmented/dataset_1M_vishing_.parquet'

print(f'Saving dataset to {output_path}...')
t0 = time.time()
df_augmented.to_parquet(output_path, engine='pyarrow', index=False)